# A chat app interface
A chat app with FastHTML and claudette/lisette

- User enters prompt, the chat bubble for the user shows.
- LLM processes. And returns with the response in an AI chat bubble.

In [ ]:
#| default_exp chat

In [ ]:
#| export
from fasthtml.common import *
from lisette import *

In [ ]:
#| hide
from IPython.display import HTML
from fasthtml.jupyter import *

## Creating the app with daisy styling -

In [ ]:
#| export
app = FastHTML(hdrs=(
        Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
        Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
        Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
        Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/@tailwindcss/typography/dist/typography.min.css'),
        MarkdownJS(),
        ),
    )

In [ ]:
#| export
# For convenience
rt = app.route

In [ ]:
#| hide
#| eval: false
srv = JupyUvi(app)

## LLM system prompt -

In [ ]:
#| export
sysp = "You are a chat model, in a chat with a user. Given the chat history as context, answer the user's current question concisely."

## Previewing components -

#| hide

Convenience function for previewing components

In [ ]:
#| hide
def preview(c):
    return HTMX(c, app=app, host=None, port=None)


In [ ]:
#| hide
@patch
def __html__(self:FT):
    return preview(self).data

## Chat with llm

In [ ]:
#| export
#| hide
class MsgType(enum.StrEnum):
    note = 'Note'
    prompt = 'Prompt'
    ai = 'AI'

In [ ]:
#| export
def bubble(msg:str, msg_type:MsgType=MsgType.prompt):
    if msg_type == MsgType.ai:
        return Div(Div(msg, cls='chat-bubble chat-bubble-secondary marked prose'), cls='chat chat-end')
    elif msg_type == MsgType.prompt:
        return Div(Div(msg, cls='chat-bubble chat-bubble-primary'), cls='chat chat-start')
    else:
        return Div(Div(msg, cls='chat-bubble chat-bubble-warning marked prose'), cls='chat chat-start')

In [ ]:
content = """
Here are some _markdown_ elements.

- This is a list item
- This is another list item
- And this is a third list item

**Fenced code blocks work here.**

## A title
Here is some text!
### Another title

"""

In [ ]:
bubble(content, MsgType.note)

<div class="chat chat-start"><div class="chat-bubble chat-bubble-warning marked prose">
Here are some _markdown_ elements.

- This is a list item
- This is another list item
- And this is a third list item

**Fenced code blocks work here.**

## A title
Here is some text!
### Another title

</div></div>

<div class="chat chat-start"><div class="chat-bubble chat-bubble-warning marked prose">
Here are some _markdown_ elements.

- This is a list item
- This is another list item
- And this is a third list item

**Fenced code blocks work here.**

## A title
Here is some text!
### Another title

</div></div>

In [ ]:
bubble("Can you help?")

<div class="chat chat-start"><div class="chat-bubble chat-bubble-primary">Can you help?</div></div>

<div class="chat chat-start"><div class="chat-bubble chat-bubble-primary">Can you help?</div></div>

In [ ]:
#| export
def mk_prompt(msg, h):
    return "Past context/conversation: " + "\n".join(h) + "\n\nCurrent question: " + msg

In [ ]:
mk_prompt("Hello", ["Old message"])

'Past context/conversation: Old message\n\nCurrent question: Hello'

'Past context/conversation: Old message\n\nCurrent question: Hello'

## Routes

In [ ]:
#| export
# We need a way to store the history of the conversation.
h = []

In [ ]:
#| export
@rt
def ask_llm(msg:str):
    global h
    r = Chat(model='claude-haiku-4-5', sp=sysp)(mk_prompt(msg, h))
    rsp = r.choices[0].message.content
    h += [f'user: {msg}, llm: {rsp}']
    return bubble(rsp, MsgType.ai)

In [ ]:
#ask_llm("What's the weather?")

In [ ]:
#| export
def llm_rsp(msg):
    return Div(
        Span(cls='loading loading-ring loading-md'), cls='flex justify-center',
        hx_post=ask_llm, hx_trigger='load', hx_swap='outerHTML',
        hx_vals=f'{{"msg": "{msg}"}}'
        )

In [ ]:
#| export
@rt
def send(msg: str, msg_type:str):
    msg_type = MsgType[msg_type]
    if msg_type == MsgType.note:
        global h
        h += [f'user_note: {msg}']
        return bubble(msg, msg_type)
    else:
        return (bubble(msg, msg_type), llm_rsp(msg))

In [ ]:
#| export
js = Script('''
document.querySelectorAll('input[name="msg_type"]').forEach(radio => {
    radio.addEventListener('change', () => {
        let input = document.getElementById('msg');
        input.placeholder = radio.value === 'note' ? 'Markdown...' : 'Ask LLM...';
        input.className = radio.value === 'note' ? 'input input-warning' : 'input input-primary';

        let button = document.getElementById('send');
        button.textContent = radio.value === 'note' ? 'Add Note' : 'Prompt LLM';
        button.className = radio.value === 'note' ? 'btn btn-warning' : 'btn btn-primary';
    });
});
'''
)

In [ ]:
#| export
@rt
def chatpg():
    return Titled("Chat Demo",
        Div(id='chat', style='max-width:600px; margin:auto; border: 1px solid #ccc; border-radius: 0.5rem; height: 400px; overflow-y: auto; padding 1rem'),
        Form(
            Label(Input(type='radio', name='msg_type', value='prompt', checked='checked', cls='radio radio-primary'), 'Prompt'),
            Label(Input(type='radio', name='msg_type', value='note', cls='radio radio-secondary'), 'Note'),
            Input(id='msg', type='text', placeholder='Type something...', style='flex:1', cls='input input-primary'),
            Button('Prompt LLM', id='send', hx_post=send, hx_target='#chat', hx_swap='beforeend', cls='btn btn-primary'),
            style='display:flex; gap:8px; max-width:600px; margin:auto; padding:1rem'
        ),
        js,
        style='text-align:center'
    )

In [ ]:
preview(chatpg)

HTML(<iframe src="/chatpg" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

HTML(<iframe src="/chatpg" style="width: 100%; height: auto; border: none;" onload="{
        let frame = this;
        window.addEventListener('message', function(e) {
            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe
            if (e.data.height) frame.style.height = (e.data.height+1) + 'px';
        }, false);
    }" allow="accelerometer; autoplay; camera; clipboard-read; clipboard-write; display-capture; encrypted-media; fullscreen; gamepad; geolocation; gyroscope; hid; identity-credentials-get; idle-detection; magnetometer; microphone; midi; payment; picture-in-picture; publickey-credentials-get; screen-wake-lock; serial; usb; web-share; xr-spatial-tracking"></iframe> )

## Make note with markdown format

In [ ]:
#| hide
#| eval: false
srv.stop()

In [ ]:
#| hide
from nbdev import nbdev_export; nbdev_export()